In [1]:
from pathlib import Path
import pandas as pd
from _bootstrap import setup
import numpy as np
import cv2


In [2]:
def obtener_nombre_archivo(ruta: Path) -> str:
    return Path(ruta).stem

In [3]:
def leer_imagen(ruta_imagen: str | Path):
    img = cv2.imread(str(ruta_imagen), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(ruta_imagen)
    return img

In [4]:
def centrar_trazo_bbox(img, tamano_salida=None):
    # Binariza para detectar el trazo como foreground
    _, bin_inv = cv2.threshold(
        img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    ys, xs = np.where(bin_inv > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None

    x0, x1 = xs.min(), xs.max() + 1
    y0, y1 = ys.min(), ys.max() + 1
    roi = img[y0:y1, x0:x1]

    if tamano_salida is None:
        H, W = img.shape
    else:
        H, W = tamano_salida

    h, w = roi.shape
    h2, w2 = min(h, H), min(w, W)
    roi = roi[:h2, :w2]

    canvas = np.full((H, W), 255, dtype=np.uint8)
    y_off = (H - h2) // 2
    x_off = (W - w2) // 2
    canvas[y_off:y_off + h2, x_off:x_off + w2] = roi
    return canvas

In [5]:
def guardar_imagen(ruta: Path, imagen):
    cv2.imwrite(ruta, imagen)

In [6]:
def procesar_imagen(
        ruta_imagen: Path,
        ruta_destino_clase: Path,
        tamano_salida=None
) -> pd.DataFrame:
    try:
        img = leer_imagen(ruta_imagen)
        nombre_original = obtener_nombre_archivo(ruta_imagen)

        filas: list[dict[str, str | float]] = []
        ruta_destino_imagen = ruta_destino_clase / f"{nombre_original}_centrado.png"

        canvas = centrar_trazo_bbox(img, tamano_salida=tamano_salida)

        guardar_imagen(ruta_destino_imagen, canvas)

        #  Path(ruta_destino_imagen).parent.mkdir(parents=True, exist_ok=True)
        if canvas is None:
            guardar_imagen(ruta_destino_imagen, img)
        else:
            guardar_imagen(ruta_destino_imagen, canvas)

        filas.append(
            {
                "archivo_origen": ruta_imagen.name,
                "archivo_generado": ruta_destino_imagen.name
            }
        )

        return pd.DataFrame(filas)

    except Exception as e:
        print(f"Error procesando {ruta_imagen}: {e}")
        return pd.DataFrame()

In [7]:
def obtener_o_crear_directorio(ruta_base: Path, carpeta_destino: str):
    ruta_completa = ruta_base / carpeta_destino

    ruta_completa.mkdir(parents=True, exist_ok=True)

    if not ruta_completa.exists():
        raise RuntimeError("No se pudo crear el directorio")

    return ruta_completa

In [8]:
# Funciones
def obtener_crear_nombre_carpeta_clase(valor_clase: str | int | float):
    """
    Devuelve el nombre de la carpeta según la 'Clase Ana':
    - Si hay número → 'clase01', 'clase02', ... 'clase11', etc.
    - Si está vacío/NaN → 'sin_clase'
    """
    if pd.isna(valor_clase) or str(valor_clase).strip() == "":
        return "sin_clase"
    try:
        numero = int(float(valor_clase))  # maneja '11.0' como 11
        return f"clase{numero:02d}"
    except ValueError:
        return "sin_clase"

In [9]:
def obtener_rutas_imagenes(ruta_origen: Path, clase: int | str, extension: str) -> list[Path]:
    nombre_carpeta = obtener_crear_nombre_carpeta_clase(clase)
    carpeta = ruta_origen / nombre_carpeta

    return list(carpeta.glob(f"*.{extension}"))

In [10]:
def procesar_clase(
        ruta_origen: Path,
        ruta_destino: Path,
        clase: int | str
) -> pd.DataFrame:
    dfs: list[pd.DataFrame] = []

    rutas_imagenes = obtener_rutas_imagenes(ruta_origen, clase, extension="png")

    if not rutas_imagenes:
        return pd.DataFrame()

    ruta_destino_clase = obtener_o_crear_directorio(
        ruta_destino,
        obtener_crear_nombre_carpeta_clase(clase)
    )

    for ruta_imagen in rutas_imagenes:
        df_perfil = procesar_imagen(ruta_imagen, ruta_destino_clase)
        dfs.append(df_perfil)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [11]:
def procesar_imagenes(
        ruta_fracciones: Path,
        ruta_centradas: Path,
        clases_a_centrar: list[int | str]
) -> pd.DataFrame:
    dfs: list[pd.DataFrame] = []

    for clase in clases_a_centrar:
        df_clase = procesar_clase(ruta_fracciones, ruta_centradas, clase)
        dfs.append(df_clase)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()



In [12]:
clases_a_centrar =  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
PATHS = setup()

csv_origen: Path = PATHS.DATA_FRACCIONES / "metadata.csv"
csv_destino: Path = PATHS.DATA_CENTRADOS / "metadata.csv"

df_fracciones = pd.read_csv(csv_origen)

df_centrados = procesar_imagenes(
    PATHS.DATA_FRACCIONES,
    PATHS.DATA_CENTRADOS,
    clases_a_centrar,
)

df_centrados
#
# df_final = construir_df_desde_imagenes(df_completos, df_mitades)
# df_final.to_csv(csv_destino, index=False)
# df_final.head()

,archivo_origen,archivo_generado
0,tu_8_1_tope_40.png,tu_8_1_tope_40_centrado.png
1,tu_8_2_base_80.png,tu_8_2_base_80_centrado.png
2,al_14b_tope_60.png,al_14b_tope_60_centrado.png
3,ba_191_4_tope_30.png,ba_191_4_tope_30_centrado.png
4,tu_8_1_base_90.png,tu_8_1_base_90_centrado.png
...,...,...
19453,al_03j_base_10.png,al_03j_base_10_centrado.png
19454,tu_96_8_tope_80.png,tu_96_8_tope_80_centrado.png
19455,ba_144_3_tope_50.png,ba_144_3_tope_50_centrado.png
19456,cc_11_3_tope_30.png,cc_11_3_tope_30_centrado.png
